# Ti MACE v3: held-out test, properties and melting point (clean version)
Training is **finished**: the models are in your Drive folder `ti_mace_v3/mod`.

| Section | Runtime needed | Time |
|---|---|---|
| 0. Setup | CPU or GPU | 2 min |
| 1. Held-out test | CPU is fine | 15–30 min (already done: Init 2.3 meV/atom, 0.006 eV/Å; Bulk 6.5 meV/atom, 0.159 eV/Å) |
| 2. Physical properties | CPU is fine | 30–90 min |
| 3. Melting point | **GPU (T4)** | about 2 h |

On a CPU runtime: run sections 0, 1 (optional, already done) and 2. When Colab gives you a GPU again: Runtime → Change runtime type → T4 GPU, then run sections 0 and 3. Every section skips work that is already saved in Drive.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch
WORK = '/content/drive/MyDrive/ti_mace_v3'
MODEL_JIT = f'{WORK}/mod/ti_mace_v3_stagetwo_compiled.model'   # loads on CPU and GPU
MODEL = f'{WORK}/mod/ti_mace_v3_stagetwo.model'                # loads on GPU only
assert os.path.exists(MODEL_JIT), 'model not found in Drive/ti_mace_v3/mod'
!mkdir -p /content/data && cd /content/data && unzip -o -q {WORK}/mace_v3.zip
!pip install -q mace-torch==0.3.17
GPU = torch.cuda.is_available()
print('GPU available:', GPU)

In [ ]:
%%writefile /content/data/mace_jit_calc.py
"""ASE calculator for a compiled (TorchScript) MACE model on CPU; works for models trained on a GPU."""
import warnings, numpy as np, torch
from ase.calculators.calculator import Calculator, all_changes
from mace import data as mdata
from mace.tools import torch_geometric, utils
warnings.filterwarnings('ignore')
class MACEJit(Calculator):
    implemented_properties = ['energy', 'free_energy', 'forces', 'stress']
    def __init__(self, path, dtype='float64', **kw):
        super().__init__(**kw)
        self.dt = torch.float64 if dtype == 'float64' else torch.float32
        torch.set_default_dtype(self.dt)
        self.model = torch.jit.load(path, map_location='cpu').eval()
        if self.dt == torch.float64: self.model = self.model.double()
        self.z = utils.AtomicNumberTable([int(z) for z in self.model.atomic_numbers]); self.rc = float(self.model.r_max)
    def calculate(self, atoms=None, properties=('energy',), system_changes=all_changes):
        super().calculate(atoms, properties, system_changes)
        cfg = mdata.config_from_atoms(atoms, key_specification=mdata.KeySpecification())
        ad = mdata.AtomicData.from_config(cfg, z_table=self.z, cutoff=self.rc)
        b = next(iter(torch_geometric.dataloader.DataLoader([ad], batch_size=1, shuffle=False))).to_dict()
        b = {k: (v.to(self.dt) if torch.is_tensor(v) and v.is_floating_point() else v) for k, v in b.items()}
        out = self.model(b, compute_force=True, compute_stress=True, training=False)
        e = float(out['energy'].detach()[0])
        self.results = {'energy': e, 'free_energy': e, 'forces': out['forces'].detach().numpy(),
                        'stress': out['stress'].detach().numpy()[0][[0, 1, 2, 1, 0, 0], [0, 1, 2, 2, 2, 1]]}


In [ ]:
%%writefile /content/data/mace_test_cpu.py
"""Held-out test of a MACE model on CPU from its compiled (TorchScript) file, which loads on a CPU-only machine
even when the model was trained on a GPU. usage: python3 mace_test_cpu.py ti_mace_v3_stagetwo_compiled.model
Reads test_Init.xyz and test_Bulk.xyz (REF_energy, REF_forces) from the current folder."""
import sys, warnings, numpy as np, torch
warnings.filterwarnings('ignore')
from ase.io import read
from mace import data as mdata
from mace.tools import torch_geometric, utils
torch.set_default_dtype(torch.float32)
model = torch.jit.load(sys.argv[1], map_location='cpu').eval()
z_table = utils.AtomicNumberTable([int(z) for z in model.atomic_numbers])
r_max = float(model.r_max)
def predict(atoms):
    cfg = mdata.config_from_atoms(atoms, key_specification=mdata.KeySpecification())
    ad = mdata.AtomicData.from_config(cfg, z_table=z_table, cutoff=r_max)
    batch = next(iter(torch_geometric.dataloader.DataLoader([ad], batch_size=1, shuffle=False))).to_dict()
    out = model(batch, compute_force=True, training=False)
    return float(out['energy'].detach()[0]), out['forces'].detach().numpy()
for g in ('Init', 'Bulk'):
    de, df = [], []
    for a in read(f'test_{g}.xyz', ':'):
        e0, f0 = a.info['REF_energy'], a.arrays['REF_forces']
        e, f = predict(a); de.append((e - e0) / len(a)); df.append((f - f0).ravel())
    print(f'{g:5s} held-out: energy RMSE {1000 * np.sqrt(np.mean(np.square(de))):6.1f} meV/atom, '
          f'force RMSE {np.sqrt(np.mean(np.square(np.concatenate(df)))):.3f} eV/A  ({len(de)} configs)', flush=True)


In [ ]:
%%writefile /content/data/mace_props_jit.py
"""Static properties of Ti with a compiled MACE model on CPU (ASE) (same strains as props.py; smaller cells,
48-atom hcp and 144-atom vacancy cell, because MACE on a CPU is slow).
usage: python mace_props_jit.py <compiled model>
Outputs JSON: hcp a, c/a, elastic constants, bcc/fcc/omega - hcp energies, vacancy formation energy."""
import sys, json, numpy as np
from ase import Atoms
from ase.build import bulk
from ase.filters import FrechetCellFilter
from ase.optimize import BFGS
from ase.units import GPa
from mace_jit_calc import MACEJit
calc = MACEJit(sys.argv[1], dtype='float64')
def relax(at, cell=True, fmax=1e-3, steps=400):
    at.calc = calc
    BFGS(FrechetCellFilter(at, hydrostatic_strain=False) if cell else at, logfile=None).run(fmax=fmax, steps=steps)
    return at
def omega(a=4.6, c=2.85):
    return Atoms('Ti3', scaled_positions=[(0, 0, 0), (1/3, 2/3, .5), (2/3, 1/3, .5)],
                 cell=[[a, 0, 0], [-a/2, a*3**.5/2, 0], [0, 0, c]], pbc=True)
out = {}
h = relax(bulk('Ti', 'hcp', a=2.93, c=4.66, orthorhombic=True).repeat((3, 2, 2)))      # 48 atoms
N = len(h); e_hcp = h.get_potential_energy() / N
a = h.cell[0, 0] / 3; c = h.cell[2, 2] / 2
out.update(hcp_a=a, hcp_c_over_a=c / a, E_hcp=e_hcp, V_hcp=h.get_volume() / N)
# elastic constants: +-0.5% strain, internal relaxation at fixed cell, central differences
d = 0.005; C = np.zeros((6, 6)); cell0 = h.cell.array.copy()
pairs = {0: (0, 0), 1: (1, 1), 2: (2, 2), 3: (1, 2), 4: (0, 2), 5: (0, 1)}
for j in range(6):
    s = []
    for sgn in (1, -1):
        eps = np.zeros((3, 3)); p, q = pairs[j]
        if p == q: eps[p, p] = sgn * d
        else: eps[p, q] = eps[q, p] = sgn * d / 2           # engineering shear strain = sgn*d
        b = h.copy(); b.set_cell(cell0 @ (np.eye(3) + eps), scale_atoms=True); relax(b, cell=False, fmax=1e-4)
        s.append(b.get_stress(voigt=True) / GPa)               # ASE: tensile positive; Voigt xx yy zz yz xz xy
    C[:, j] = (s[0] - s[1]) / (2 * d)
C = 0.5 * (C + C.T)
out.update({'C11': C[0, 0], 'C12': C[0, 1], 'C13': C[0, 2], 'C33': C[2, 2], 'C44': C[3, 3], 'C66': C[5, 5]})
for ph, at in (('bcc', bulk('Ti', 'bcc', a=3.25, cubic=True)), ('fcc', bulk('Ti', 'fcc', a=4.1, cubic=True)), ('omega', omega())):
    at = relax(at); e = at.get_potential_energy() / len(at)
    out[f'E_{ph}'] = e; out[f'dE_{ph}-hcp_meV'] = (e - e_hcp) * 1000
# vacancy: 4x3x3 orthorhombic hcp (144 atoms), relaxed at constant volume
v = relax(bulk('Ti', 'hcp', a=a, c=c, orthorhombic=True).repeat((4, 3, 3)), cell=False); n = len(v); e0 = v.get_potential_energy() / n
del v[0]; relax(v, cell=False)
out['E_vac_eV'] = v.get_potential_energy() - (n - 1) * e0
res = {k: round(float(x), 4) for k, x in out.items()}
print(json.dumps(res, indent=1)); json.dump(res, open('mace_v3_props.json', 'w'), indent=1)


In [ ]:
%%writefile /content/data/mace_melt.py
"""Melting point of Ti from solid-liquid coexistence with a MACE model (ASE, GPU).
bcc slab (6x6x14 cells, 1008 atoms): equilibrate at T, melt the upper half at 3000 K with the lower half fixed,
then run the two-phase box at T and zero pressure (Berendsen) for t_coex ps. If the potential energy falls, the
liquid crystallises (T < Tm); if it rises, the solid melts (T > Tm). usage: mace_melt.py model T [t_coex_ps] [device]"""
import sys, numpy as np
from ase.build import bulk
from ase import units
from ase.constraints import FixAtoms
from ase.md.langevin import Langevin
from ase.md.nptberendsen import NPTBerendsen
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
model, T = sys.argv[1], float(sys.argv[2]); tco = float(sys.argv[3]) if len(sys.argv) > 3 else 30.0
dev = sys.argv[4] if len(sys.argv) > 4 else 'cuda'; REP = (6, 6, 14) if dev == 'cuda' else (2, 2, 4)
dt = 2.0 * units.fs; kappa = 1 / (110 * units.GPa)                  # compressibility for the barostat
at = bulk('Ti', 'bcc', a=3.28, cubic=True).repeat(REP)
at.calc = MACECalculator(model_paths=model, device=dev, default_dtype='float32')
MaxwellBoltzmannDistribution(at, temperature_K=T, rng=np.random.default_rng(1))
npt = lambda: NPTBerendsen(at, dt, temperature_K=T, taut=100 * units.fs, pressure_au=0.0, taup=1000 * units.fs,
                           compressibility_au=kappa)
npt().run(2500)                                                      # 5 ps solid at T, zero pressure
z = at.positions[:, 2]; top = z > 0.5 * at.cell[2, 2]
at.set_constraint(FixAtoms(mask=~top))
Langevin(at, dt, temperature_K=3000, friction=0.02, rng=np.random.default_rng(2)).run(2000)   # 4 ps: melt the top
at.set_constraint(); MaxwellBoltzmannDistribution(at, temperature_K=T, rng=np.random.default_rng(3))
md = npt(); out = []
def log(): out.append((len(out) * 0.1, at.get_potential_energy() / len(at), at.get_temperature(), at.get_volume() / len(at)))
md.attach(log, interval=50); md.run(int(tco * 500))
d = np.array(out); np.savetxt(f'melt_T{T:.0f}.dat', d, header='t_ps pe_eV_per_atom T_K V_A3_per_atom')
n = len(d) // 3; slope = np.polyfit(d[n:, 0], d[n:, 1], 1)[0] * 1000
print(f'T = {T:.0f} K: dE/dt over the last 2/3 = {slope:+.3f} meV/atom/ps -> '
      f'{"solid grows (T < Tm)" if slope < 0 else "solid melts (T > Tm)"};  V = {d[-1, 3]:.2f} A3/atom')


## 1. Held-out test (CPU is fine; skipped if already saved)

In [ ]:
%cd /content/data
if os.path.exists(f'{WORK}/test_errors_v3.txt'):
    print(open(f'{WORK}/test_errors_v3.txt').read())
else:
    !python mace_test_cpu.py {MODEL_JIT} | grep held-out | tee {WORK}/test_errors_v3.txt

## 2. Physical properties (CPU is fine; skipped if already saved)
Lattice parameters, elastic constants, bcc/fcc/ω − hcp energies, vacancy formation energy: same cells and strains as the earlier MACE column of Table 2.

In [ ]:
%cd /content/data
if os.path.exists(f'{WORK}/mace_v3_props.json'):
    print(open(f'{WORK}/mace_v3_props.json').read())
else:
    !python mace_props_jit.py {MODEL_JIT} && cp mace_v3_props.json {WORK}/

## 3. Melting point (GPU only; finished temperatures are skipped)

In [ ]:
%cd /content/data
if not GPU:
    print('No GPU in this session: change the runtime to T4 GPU and run sections 0 and 3.')
else:
    for T in (1700, 1800, 1900, 2000, 2100):
        if os.path.exists(f'{WORK}/melt_T{T}.dat'):
            print(f'T = {T} K already done'); continue
        !python mace_melt.py {MODEL} {T} 30 cuda 2>&1 | grep "T = " | tee -a {WORK}/melting_v3.txt
        !cp melt_T{T}.dat {WORK}/

## 4. Summary

In [ ]:
for f in ('test_errors_v3.txt', 'mace_v3_props.json', 'melting_v3.txt'):
    p = f'{WORK}/{f}'
    print(f'--- {f}'); print(open(p).read() if os.path.exists(p) else '(not yet)')